# ViralCut AI — Google Colab (T4)

**Runtime → Change runtime type → GPU (Tesla T4)** then **Runtime → Run all**.
This notebook installs dependencies, verifies the GPU, starts the FastAPI backend,
builds the React frontend, exposes the app with a Cloudflare Quick Tunnel and prints the URL.

In [ ]:
import os
import shutil
import subprocess
import sys
import time

REPO = "ViralCut-AI"
REPO_URL = "https://github.com/your-org/ViralCut-AI.git"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("Working in", os.getcwd())

## 1. Verify GPU

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install dependencies

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install -r requirements.txt
import faster_whisper, yt_dlp, fastapi
print("Dependencies OK")
print("FFmpeg:", shutil.which("ffmpeg"))

## 3. Configure environment

Set your OpenRouter API key (get one at https://openrouter.ai/keys).

In [ ]:
OPENROUTER_API_KEY = ""  # <-- paste your key here

with open(".env", "w") as f:
    f.write(f"OPENROUTER_API_KEY={OPENROUTER_API_KEY}\n")
    f.write("OPENROUTER_MODEL=openrouter/auto\n")
    f.write("WHISPER_MODEL=medium\n")
    f.write("WHISPER_DEVICE=cuda\n")
    f.write("WHISPER_COMPUTE_TYPE=float16\n")
    f.write("MAX_CLIPS=5\n")
    f.write("MIN_CLIP_DURATION=20\n")
    f.write("MAX_CLIP_DURATION=60\n")

print("Config written." if OPENROUTER_API_KEY else "WARNING: no API key set yet — analysis will fail until you add it.")

## 4. Verify device detection

In [ ]:
from app.config import settings, is_cuda_available
from app.clipper import detect_nvenc

print("CUDA available:", is_cuda_available())
print("Whisper device:", settings.whisper_device, "| compute:", settings.whisper_compute_type)
print("NVENC available:", detect_nvenc())

## 5. Start backend (background)

In [ ]:
import threading
import uvicorn

def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(4)

!curl -s http://localhost:8000/api/devices
print()
print("Backend listening on :8000")

## 6. Build frontend

In [ ]:
%cd web
!npm install --no-audit --no-fund > /dev/null 2>&1
!npm run build
%cd ..
print("Frontend built into web/dist — served by FastAPI at / ")

## 7. Expose publicly (Cloudflare Quick Tunnel)

In [ ]:
!which cloudflared || wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import subprocess

p = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for _ in range(60):
    line = p.stdout.readline()
    if not line:
        break
    if "trycloudflare.com" in line:
        url = line.split("|")[-1].strip()
        break

print("\n========== VIRALCUT AI IS LIVE ==========")
print(url or "(tunnel URL not found yet — check output above)")
print("==========================================")
print("Local:  http://localhost:8000")